# Обработка текстов

Задача — по поисковому запросу найти подходящие объявления услуг. Данные (размеры таблиц — в `ingestion/raw.ipynb`):

- **корпус** — 189 212 объявлений из `benchmark_items.parquet`; только среди них выбираем 50 ответов на каждый тестовый запрос;
- **тестовые запросы** — 2 452 запроса из `benchmark_queries.parquet`, правильных ответов к ним нет;
- **train** — 497 673 строки прошлых поисков: поля поиска (текст запроса, локация, фильтры, категория, флаг доставки) и объявление, которое пользователь в итоге выбрал. Выбранные в train объявления не обязательно есть в корпусе.

У объявления три текстовых поля: **заголовок**, **параметры** и **описание**. Параметры — это поля анкеты объявления (вид услуги, адрес, цены, график работы), записанные одной строкой подряд: название поля, за ним значение, например «Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу». У запроса два текстовых поля: **текст запроса** и **фильтры**. Фильтры — это значения, которые пользователь выбрал в поиске, например «Вид услуги Красота, здоровье»; записаны они так же, как параметры объявления. Выбирают их не всегда, поэтому у части запросов фильтров нет.

Объявления ищем двумя способами:

- **BM25** — стандартная формула поиска по словам: объявление получает тем больше очков, чем больше в нём слов запроса и чем реже эти слова встречаются в других объявлениях корпуса (подробнее — `retrieval/baseline.ipynb`);
- **поиск по смыслу** — нейросеть-**эмбеддер** превращает текст в вектор (набор чисел) так, что у текстов с похожим смыслом векторы близки, и мы берём объявления, чьи векторы ближе всего к вектору запроса (подробнее — `embeddings/models.ipynb`). Совпадения слов для этого не нужно.

Этот ноутбук — про подготовку текстов для BM25. Чтобы искать по словам, тексты нужно привести к одному виду: «Автоподбора», «автоподбор» и «АВТОПОДБОР» должны считаться одним и тем же словом. Это делает `avito.processing.text` (запуск — `scripts/processing/process.py`):

1. переводит текст в нижний регистр и заменяет «ё» на «е»;
2. разбивает текст на слова: словом считаются подряд идущие буквы и цифры, всё остальное — разделители;
3. выбрасывает служебные слова: предлоги, союзы и частицы вроде «в», «на», «и», «для», «не»;
4. отрезает окончания (это называется стеммингом) с помощью алгоритма Snowball из библиотеки PyStemmer: «автомобиля» → «автомобил», «Москве» → «москв».

Так обрабатываются заголовок, параметры и описание объявлений, а у запросов — текст запроса и фильтры. Эмбеддер получает исходный текст без этой обработки.

In [1]:
import polars as pl

from avito.embeddings.texts import DESCRIPTION_CHARS, normalize_spaces
from avito.ingestion import load as ingested
from avito.processing import load as processed
from avito.processing.text import stem, tokenize

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_width_chars(220);

## 1. Как это выглядит

Один текст по шагам:

In [2]:
text = "Автоподбор, разовый осмотр автомобиля в Москве и области! Ёлки-палки 4х4"

pl.DataFrame({
    "шаг": ["исходный текст", "слова без служебных", "после отрезания окончаний"],
    "результат": [text, " ".join(tokenize(text)), stem(text)],
})

шаг,результат
str,str
"""исходный текст""","""Автоподбор, разовый осмотр автомобиля в Москве и области! Ёлки-палки 4х4"""
"""слова без служебных""","""автоподбор разовый осмотр автомобиля москве области елки палки 4х4"""
"""после отрезания окончаний""","""автоподбор разов осмотр автомобил москв област елк палк 4х4"""


Первые три объявления корпуса — заголовок и параметры до и после обработки:

In [3]:
raw = ingested.load_corpus(["item_id", "item_title_raw", "item_infm_params_text"]).head(3)
stems = processed.load_corpus(["item_id", "title_stems", "params_stems"]).head(3)

raw.join(stems, on="item_id").select(
    pl.col("item_title_raw").alias("заголовок"),
    pl.col("title_stems").alias("заголовок после обработки"),
    pl.col("item_infm_params_text").alias("параметры"),
    pl.col("params_stems").alias("параметры после обработки"),
)

заголовок,заголовок после обработки,параметры,параметры после обработки
str,str,str,str
"""Ремонт/выкуп компьют. и ноутбуков с выездом на дом""","""ремонт выкуп компьют ноутбук выезд дом""","""Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от …","""вид услуг компьютерн помощ мест оказан услуг пр т ленин тип стоимост услуг начальн цен график работ 34200 график работ 8…"
"""Обучение ребенка чтению""","""обучен ребенк чтен""","""Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, логопеды Тип стоимост…","""вид услуг обучен курс мест оказан услуг ул чернышевск 35 тип услуг детск развит логопед тип стоимост урок начальн цен гр…"
"""Афрокудри 5+""","""афрокудр 5""","""Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип услуги Услуги парикмахе…","""вид услуг красот здоров мест оказан услуг городск округ иванов фрунзенск район тип услуг услуг парикмахер тип стоимост у…"


И первые восемь тестовых запросов — текст запроса и фильтры до и после обработки. Где пользователь не выбирал фильтров, колонка пустая:

In [4]:
processed.load_bench_queries().select(
    pl.col("search_query").alias("запрос"),
    pl.col("query_stems").alias("запрос после обработки"),
    pl.col("search_infm_params_text").alias("фильтры"),
    pl.col("filter_stems").alias("фильтры после обработки"),
).head(8)

запрос,запрос после обработки,фильтры,фильтры после обработки
str,str,str,str
"""перевозки владикавказ тбилиси""","""перевозк владикавказ тбилис""","""""",""""""
"""обзвон по базе""","""обзвон баз""","""Вид услуги Деловые услуги""","""вид услуг делов услуг"""
"""липоредукция подбородка""","""липоредукц подбородк""","""Вид услуги Красота, здоровье""","""вид услуг красот здоров"""
"""подъемник 4 х стоечный""","""подъемник 4 х стоечн""","""""",""""""
"""монтаж видеодомофонов""","""монтаж видеодомофон""","""""",""""""
"""боевое самбо""","""боев самб""","""Тип услуги Дизайн, рисование Вид услуги Обучение, курсы""","""тип услуг дизайн рисован вид услуг обучен курс"""
"""услуги бухгалтера для ип""","""услуг бухгалтер ип""","""Вид услуги Деловые услуги Тип услуги Бухгалтерия, финансы""","""вид услуг делов услуг тип услуг бухгалтер финанс"""
"""бригада по установке забора из профнастила""","""бригад установк забор профнаст""","""""",""""""


## 2. Где в объявлении находятся слова запроса

Чтобы понять, по каким полям объявления искать, берём пары из train «текст запроса → выбранное объявление» и смотрим, какая доля слов запроса встречается в выбранном объявлении. Фильтры здесь не учитываем, только текст запроса.

В train частые тексты повторяются много раз: 100 самых частых текстов дают четверть всех строк. А в тестовых запросах все 2 452 текста разные (`ingestion/insights.ipynb`). Если брать все строки train, частые тексты перевесили бы остальные. Поэтому, как в тесте, берём каждый текст один раз: случайные 30 000 разных текстов, и для каждого одно выбранное объявление — из первой строки train с этим текстом.

Выбранные объявления берём из таблицы всех объявлений (`load_items`), а не только из корпуса (`load_corpus`): объявлений, выбранных в train, в корпусе есть лишь часть (`ingestion/raw.ipynb`). Берём обработанные поля, в которых ищем слова, и исходные заголовок и описание: заголовок нужен для примеров, а от исходного описания ниже отрезаем начало.

In [5]:
train = processed.load_train(["search_query", "query_stems", "item_id"])
texts = train.unique("search_query", keep="first", maintain_order=True).sample(30_000, seed=0)
items = processed.load_items(["item_id", "title_stems", "params_stems", "description_stems"])
raw_items = ingested.load_items(["item_id", "item_title_raw", "item_description_raw"])
pairs = texts.join(items, on="item_id").join(raw_items, on="item_id")

pl.DataFrame({"пар запрос — выбранное объявление": [pairs.height]})

пар запрос — выбранное объявление
i64
30000


Для каждой пары считаем, какая доля разных слов запроса нашлась в объявлении (1 — нашлись все слова, 0 — ни одного). Сравниваем четыре варианта: только заголовок; заголовок и параметры; заголовок, параметры и первые 700 символов описания; заголовок, параметры и всё описание. Вариант с 700 символами нужен, чтобы проверить, можно ли брать только начало описания: эмбеддер получает из описания как раз первые 700 символов (раздел 3).

In [6]:
def found_share(query: str, field: str) -> float:
    words = set(query.split())
    if not words:
        return 0.0
    return len(words & set(field.split())) / len(words)


rows = []
for row in pairs.iter_rows(named=True):
    title = row["title_stems"]
    title_params = title + " " + row["params_stems"]
    description_start = stem(normalize_spaces(row["item_description_raw"])[:DESCRIPTION_CHARS])
    rows.append({
        "заголовок": found_share(row["query_stems"], title),
        "заголовок и параметры": found_share(row["query_stems"], title_params),
        "заголовок, параметры и первые 700 символов описания": found_share(row["query_stems"], title_params + " " + description_start),
        "заголовок, параметры и всё описание": found_share(row["query_stems"], title_params + " " + row["description_stems"]),
    })
shares = pl.DataFrame(rows)

pl.DataFrame({
    "где ищем слова запроса": shares.columns,
    "доля пар, где нашлись все слова": [(shares[column] == 1).mean() for column in shares.columns],
    "доля пар, где нашлось хотя бы одно слово": [(shares[column] > 0).mean() for column in shares.columns],
    "доля пар, где не нашлось ни одного слова": [(shares[column] == 0).mean() for column in shares.columns],
    "средняя доля найденных слов запроса": [shares[column].mean() for column in shares.columns],
})

где ищем слова запроса,"доля пар, где нашлись все слова","доля пар, где нашлось хотя бы одно слово","доля пар, где не нашлось ни одного слова",средняя доля найденных слов запроса
str,f64,f64,f64,f64
"""заголовок""",0.160433,0.762867,0.237133,0.456002
"""заголовок и параметры""",0.268067,0.859333,0.140667,0.575195
"""заголовок, параметры и первые 700 символов описания""",0.460967,0.929,0.071,0.717797
"""заголовок, параметры и всё описание""",0.607167,0.9515,0.0485,0.798948


Только у 16% пар все слова запроса есть в заголовке выбранного объявления. Если добавить параметры — у 27%, если добавить первые 700 символов описания — у 46%, а со всем описанием — у 61%. Значит, искать надо по всем трём полям, и описание нужно брать целиком: если брать только его начало, пар, где нашлись все слова, становится заметно меньше.

У 4.9% пар во всём выбранном объявлении нет ни одного слова запроса. Такие объявления BM25 по тексту запроса не найдёт: их оценка по словам запроса равна нулю. Десять случайных таких пар:

In [7]:
no_common = pairs.filter(shares["заголовок, параметры и всё описание"] == 0)

no_common.sort("search_query").sample(10, seed=0).select(
    pl.col("search_query").alias("запрос"),
    pl.col("query_stems").alias("запрос после обработки"),
    pl.col("item_title_raw").alias("заголовок выбранного объявления"),
    pl.col("title_stems").alias("заголовок после обработки"),
)

запрос,запрос после обработки,заголовок выбранного объявления,заголовок после обработки
str,str,str,str
"""кладчики на пеноблок""","""кладчик пеноблок""","""Строительство домов из теплоблока""","""строительств дом теплоблок"""
"""лениногорск сауна""","""лениногорск саун""","""Романтическое свидание для двоих""","""романтическ свидан дво"""
"""краснодеревщик""","""краснодеревщик""","""Работы на станках с чпу, раскрой, резьба по дереву""","""работ станк чпу раскр резьб дерев"""
"""darricake""","""darricake""","""Мотор тестер""","""мотор тестер"""
"""нянч""","""нянч""","""Профессиональная няня с 0+""","""профессиональн нян 0"""
"""suzuki jimny 2025""","""suzuki jimny 2025""","""AvtokeyNK42 (Автоключи и чипы)""","""avtokeynk42 автоключ чип"""
"""схемы организации дорожного движения""","""схем организац дорожн движен""","""Чертежи в компас, SolidWorks""","""чертеж компас solidworks"""
"""стоматолог с выездом""","""стоматолог выезд""","""Ремонт. Коронки. Протезы""","""ремонт коронк протез"""
"""ип бурыкин бытовки""","""ип бурыкин бытовк""","""Изготовление бытовок,модульные домиков и бань""","""изготовлен бытовок модульн домик бан"""


Среди них есть близкие по смыслу слова («афроплетение» → «Афропрически любой сложности», «стоматолог с выездом» → «Ремонт. Коронки. Протезы»), опечатки («нянч» → «Профессиональная няня») и разные формы одного слова, которые при отрезании окончаний не свелись к одной основе («бытовки» стало «бытовк», а «бытовок» осталось «бытовок»). У некоторых пар связь по тексту не видна совсем («darricake» → «Мотор тестер»). Близкие по смыслу пары и, вероятно, часть опечаток может найти поиск по смыслу, поэтому он и нужен в решении как второй способ поиска.

## 3. Шаблонные слова в параметрах

Параметры — это названия полей анкеты вперемешку с их значениями (см. начало ноутбука), и многие названия полей есть почти в каждом объявлении: «Место оказания услуг», «Вид услуги», «Тип стоимости». Смотрим, в какой доле объявлений корпуса встречается каждое слово параметров после обработки. Показаны десять самых частых слов:

In [8]:
corpus_params = processed.load_corpus(["params_stems"])
words = corpus_params.select(pl.col("params_stems").str.split(" ").list.unique().alias("слово после обработки")).explode("слово после обработки", empty_as_null=True)
word_counts = words.group_by("слово после обработки").len("объявлений, где есть слово").sort("объявлений, где есть слово", descending=True)
word_counts = word_counts.with_columns((pl.col("объявлений, где есть слово") / corpus_params.height).alias("доля объявлений"))

word_counts.head(10)

слово после обработки,"объявлений, где есть слово",доля объявлений
str,u32,f64
"""мест""",189102,0.999419
"""вид""",188953,0.998631
"""услуг""",187370,0.990265
"""оказан""",187330,0.990053
"""тип""",187315,0.989974
"""стоимост""",175196,0.925924
"""цен""",157827,0.834128
"""начальн""",157229,0.830967
"""работ""",152452,0.805721


Самые частые слова — обрезки названий полей: «мест» и «оказан» из «Место оказания услуг», «вид» из «Вид услуги», «тип» и «стоимост» из «Тип стоимости», «услуг» — из первых двух названий. Первые пять слов есть в 99% объявлений корпуса, «стоимост» — в 93%. BM25 учитывает, насколько слово редкое: слово, которое есть почти в каждом объявлении, почти ничего не добавляет к оценке. Поэтому для BM25 параметры оставляем как есть. Поиск BM25 по очищенным параметрам мы отдельно не проверяли.

С поиском по смыслу иначе. Эмбеддер читает не больше 256 токенов (токен — слово или часть слова), а всё, что дальше, отбрасывает. В тексте объявления для эмбеддера параметры стоят сразу после заголовка, перед описанием, и в среднем занимают почти 1000 символов (`embeddings/models.ipynb`). Такой шаблон, вероятно, занял бы большую часть этих 256 токенов, и до описания модель почти не дочитывала бы. Поэтому для эмбеддера параметры очищаются (`avito.processing.params`): остаются только значения полей вроде «Вид услуги», «Услуга», «Чем вы занимаетесь» (например, «Компьютерная помощь», «Диагностика компьютера»), а адреса, цены, графики работы и остальные поля выбрасываются. Пример параметров до и после очистки — в `embeddings/models.ipynb`.

## Выводы

- Одного заголовка мало: все слова запроса есть в заголовке выбранного объявления только у 16% пар, вместе с параметрами — у 27%, с первыми 700 символами описания — у 46%, а со всем описанием — у 61%. Поэтому BM25 ищет по заголовку, параметрам и полному описанию.
- У 4.9% пар в выбранном объявлении нет ни одного слова запроса: это близкие по смыслу слова, опечатки, формы слов, которые не свелись к одной основе, и пары без видимой связи. BM25 по тексту запроса их не найдёт, поэтому кроме BM25 в решении есть второй способ поиска — поиск по смыслу.
- Шаблонные слова параметров есть почти в каждом объявлении и почти ничего не весят в BM25, поэтому для BM25 параметры оставляем как есть. Эмбеддер читает только начало текста, поэтому для него из параметров оставляем только то, чем занимается исполнитель.
- Сама обработка (нижний регистр, выбрасывание служебных слов, отрезание окончаний) — стандартный шаг для поиска по словам. Сравнения BM25 с ней и без неё в ноутбуках проекта нет.